# Activity explorer

Owner-run, local-only stream inspection. Choose an activity ID below. Outputs may contain private health data; never commit or share them. No map tiles or network services are used.

In [ ]:
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

from stridelog.db.models import Activity, ActivityAnalytics, ActivityRecord
from stridelog.notebook_data import read_table
from stridelog.presentation import format_duration, pace_seconds_per_km, pace_ticks

In [ ]:
activities = read_table(Activity, ["id", "name", "local_date"])
pl.DataFrame(activities, schema={"id": pl.Int64, "name": pl.String, "local_date": pl.Date}).sort(
    "local_date", descending=True
)

In [ ]:
activity_id = 0  # Replace with a local activity ID.
records = read_table(
    ActivityRecord,
    ["record_index", "ts_utc", "distance_m", "speed_mps", "heart_rate_bpm", "altitude_m"],
    ActivityRecord.activity_id == activity_id,
)
if not records:
    raise ValueError("No samples found for that activity ID; check it and rebuild the archive.")
[
    {
        **{key: value for key, value in record.items() if key != "speed_mps"},
        "pace_min_per_km": (
            format_duration(pace)
            if (pace := pace_seconds_per_km(record["speed_mps"])) is not None
            else None
        ),
    }
    for record in records[:5]
]

In [ ]:
analytics = read_table(ActivityAnalytics, ["metrics"], ActivityAnalytics.activity_id == activity_id)
if not analytics:
    raise ValueError("No cached analytics found for that activity; run stridelog analyze first.")
splits = analytics[0]["metrics"]["splits"]
pl.DataFrame(splits, schema={"index": pl.Int64, "distance_m": pl.Float64, "duration_s": pl.Float64})

In [ ]:
stream = pl.DataFrame(records).sort("record_index")
paces = [pace_seconds_per_km(value) for value in stream["speed_mps"].to_list()]
figure = make_subplots(
    rows=2, cols=1, shared_xaxes=True, subplot_titles=["Pace (min/km)", "Heart rate (bpm)"]
)
figure.add_trace(
    go.Scatter(
        x=stream["distance_m"].to_list(),
        y=paces,
        customdata=[None if pace is None else format_duration(pace) for pace in paces],
        name="Pace",
        hovertemplate="Distance: %{x:.0f} m<br>Pace: %{customdata} /km<extra></extra>",
    ),
    row=1,
    col=1,
)
figure.add_trace(
    go.Scatter(
        x=stream["distance_m"].to_list(), y=stream["heart_rate_bpm"].to_list(), name="Heart rate"
    ),
    row=2,
    col=1,
)
tick_values, tick_labels = pace_ticks(paces)
if tick_values:
    figure.update_yaxes(tickmode="array", tickvals=tick_values, ticktext=tick_labels, row=1, col=1)
figure.update_yaxes(autorange="reversed", row=1, col=1)
figure.update_xaxes(title_text="Distance (m)", row=2, col=1)
figure